In [1]:
import dotenv
dotenv.load_dotenv()

True

In [2]:
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
import requests

c:\Users\koyel\Downloads\python_practice\ai_ml_learning_journey\nitish_singh\genai_notebooks\learning_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
from langchain_community.tools import DuckDuckGoSearchRun

# step 1: creating tool object
search_tool = DuckDuckGoSearchRun()
results = search_tool.invoke('Top news in India today')

C:\Users\koyel\AppData\Local\Temp\ipykernel_10888\1101394234.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


In [4]:
results

'Latest News Today: Follow today’s news updates and recent stories from India and across the globe, covering politics, business, technology, and sports. Get the latest India news, breaking headlines, political developments, government updates, weather alerts, crime news and state coverage at CNBC TV18. Get the latest news and breaking news today from India and around the world. Zee News covers politics, business, technology, entertainment, sports, cricket, festivals, and current affairs with live updates and in-depth reports. Stay updated with the latest breaking news, live updates, photos, videos and top trending stories from India and around the world across politics, economy, sports and more on The Economic Times. The latest news headlines on Life Style, Business, Politics, Hollywood and News from across the world at The Times of India.'

In [5]:
llm = ChatOpenAI()

In [6]:
llm.invoke('hi')

AIMessage(content='Hello! How can I assist you today?', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 9, 'prompt_tokens': 8, 'total_tokens': 17, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DzjcNs1QQXIFcRL5esJrnM28Xiywr', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019f472f-aada-75e0-be3f-a7d6f2b13837-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 8, 'output_tokens': 9, 'total_tokens': 17, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [17]:
from langchain_classic.agents import create_react_agent, AgentExecutor
from langsmith.client import Client

- `create_react_agent` now is completely deprecated from `langchain`. It should be used from `langgraph` now.
- But here as learning concepts from Campussx videos, just using it to finish the learning.

In [18]:
# step 2: pull the ReAct prompt from LangSmith
client = Client()
prompt = client.pull_prompt("hwchase17/react", dangerously_pull_public_prompt=True)

- react = reasoning + action.
- Agent which will reason and then take action -> then again reason and take action and it will continue until it reaches the final answer - is called `ReAct` agent. 
- This prompt will be used by ReAct agent, so this is named as react prompt.

https://smith.langchain.com/hub - link to langchainhub for prompts.

In [19]:
# step 3: Create the ReAct agent manually with the pulled prompt - it will think+plan
agent = create_react_agent(
    llm = llm,
    tools = [search_tool],
    prompt = prompt
)

- There are other design patterns as well apart from react (which is used here) - refer doc - paste link (TODO).

In [20]:
# step 4: Wrap it with AgentExecutor - executes plan on ground.
agent_executor = AgentExecutor(
    agent = agent,
    tools = [search_tool],
    verbose = True
)

In [21]:
# step 5: Invoke
response = agent_executor.invoke({'input': "3 ways to reach goa from delhi"})
print(response)



> Entering new AgentExecutor chain...
You may need to consider different modes of transportation and routes when planning a trip from Delhi to Goa.
Action: duckduckgo_search
Action Input: "best ways to reach Goa from Delhi"There are 11 ways to get from Delhi to Goa by plane, train, bus, car, or bus (Flixbus) Select an option below to see step-by-step directions and to compare ticket prices and travel times in Rome2Rio's travel planner. The fastest way to reach Goa from Delhi is by Flight which would take approximately 3 hours. The cheapest way to reach Goa from Delhi is by Train which would take approximately 29 hours. Plan your perfect Delhi to Goa trip with our guide. Find the best packages, cheapest flights, distance, cost breakdown and travel options by train and road. May 9, 2025 · Looking to escape Delhi and soak up the sun in Goa? This guide explains easy ways to travel from Delhi to Goa by air, train, and road. Find out what to expect on each route, from timing to comfort and

In [22]:
response['output']

'The 3 ways to reach Goa from Delhi are by plane (fastest), by train (cheapest), and by bus or car.'

Let's see another example.

In [23]:
response = agent_executor.invoke({'input': "Give a preview for today's IPL match between CSK and PBKS"}) 
print(response)



> Entering new AgentExecutor chain...
I need to find the latest information on the upcoming IPL match between CSK and PBKS.
Action: duckduckgo_search
Action Input: CSK vs PBKS IPL match preview1 day ago - CSK vs PBKS • 7th Match • Indian Premier League 2026 · Series · Indian Premier League 2026 · Date · Friday, April 3 · Time · 7:30 PM LOCAL, 2:00 PM GMT, 7:00 AM PT, 10:00 AM ET · Toss · Punjab Kings won the toss and opt to Bowl · Venue ... April 4, 2026 - CSK Vs PBKS, IPL 2026: Check preview, toss update, playing XIs and other details of the Chennai Super Kings vs Punjab Kings match 7 of the Indian Premier League season 19 April 3, 2026 - CSK vs PBKS Today Match Prediction – Check out who will win today’s IPL 2026, Match 7 between Chennai Super Kings and Punjab Kings, only on CricTracker.com April 3, 2026 - CSK vs PBKS IPL 2026 preview: Check match time, venue, live streaming details, predicted playing XIs, pitch report, head-to-head record, and latest team news from Chepauk. April 

Let's specify question more.

In [24]:
response = agent_executor.invoke({'input': "Give a detailed preview for today's IPL match between CSK and PBKS"}) # Give a detailed preview
print(response)



> Entering new AgentExecutor chain...
I should use the duckduckgo_search tool to find the latest information on the IPL match between CSK and PBKS.
Action: duckduckgo_search
Action Input: "CSK vs PBKS IPL match preview today"Apr 2, 2026 · IPL 2026 CSK vs PBKS Match Preview: Shreyas Iyer has been declared fit for the clash, while CSK are likely to be without Dewald Brevis and MS Dhoni Apr 3, 2026 · CSK vs PBKS Dream11 Team Today - Find out Chennai Super Kings vs Punjab Kings Dream11 prediction, playing XI, captain, vice-captain and more updates for Match 7 of IPL, only on... Apr 3, 2026 · The CSK vs PBKS clash will be held at the MA Chidambaram Stadium in Chennai, with the match set to begin at 7:30 PM IST. The game will be broadcast on the Star Sports Network, while live streaming ... Apr 2, 2026 · CSK vs PBKS IPL 2026 match preview with full details on playing XIs, pitch report, weather, head-to-head stats, key players, predictions, and live streaming info. Apr 3, 2026 · Get the bes

In [25]:
import os

WEATHERSTACK_API_KEY = os.getenv("WEATHERSTACK_API_KEY")

- Link to weather api doc - https://docs.apilayer.com/weatherstack/docs/api-documentation?utm_source=dashboard&utm_medium=Referral
- Create a free account and get API Key.

In [26]:
# tool addition
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
import requests


@tool
def get_weather_data(city: str) -> str:
    """
    This function fetches the current weather data for a given city
    """
    url = f"http://api.weatherstack.com/current?access_key={WEATHERSTACK_API_KEY}&query={city}"
    response = requests.get(url)
    return response.json

In [27]:
llm = ChatOpenAI()

In [29]:
from langchain_classic.agents import create_react_agent, AgentExecutor
from langsmith.client import Client

# pull the standard ReAct agent prompt from LangSmith
client = Client()
prompt = client.pull_prompt("hwchase17/react", dangerously_pull_public_prompt=True)

agent = create_react_agent(
    llm = llm,
    tools = [search_tool, get_weather_data],
    prompt = prompt
)

In [30]:
agent_executor = AgentExecutor(
    agent = agent,
    tools = [search_tool, get_weather_data],
    verbose = True
)

In [31]:
response = agent_executor.invoke({'input': "Find the capital of Madhya Pradesh, then find its current weather condition."})
print(response)



> Entering new AgentExecutor chain...
I should first find the capital of Madhya Pradesh and then check the current weather condition there.
Action: duckduckgo_search
Action Input: "capital of Madhya Pradesh"3 weeks ago - Its capital is Bhopal. Other major cities include Indore, Gwalior, Jabalpur, Chhindwara and Sagar. Madhya Pradesh is the second largest Indian state by area and the fifth largest state by population with over 72 million residents. It borders the states of Rajasthan to the northwest, Uttar Pradesh ... 1 month ago - After the formation of Madhya Pradesh, Bhopal was part of the Sehore district. It was bifurcated in 1972 and a new district, Bhopal, was formed. Flourishing around 1707, the city was the capital of the former Bhopal State, a princely state of the British ruled by the Nawabs of Bhopal until ... 1 day ago - It has no coastline and no international frontier. Its physiography is characterized by low hills, extensive plateaus, and river valleys. The capital is B